# Stock Price Prediction using LSTM
**Student Project**

Hi! In this project, I am building a machine learning model to predict the stock price using historical data. I am using an LSTM (Long Short-Term Memory) neural network which is great for time series data like stocks.

I will also add Moving Average (MA) and Relative Strength Index (RSI) indicators to help understand the trends.

In [ ]:
# Import all the necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout
import math

import warnings
warnings.filterwarnings('ignore')

### 1. Fetching the Data
I'm using `yfinance` to get the stock data. Let's look at Apple (AAPL).

In [ ]:
stock_symbol = 'AAPL'
print(f"Downloading stock data for {stock_symbol}...")
data = yf.download(stock_symbol, start='2015-01-01', end='2023-01-01')

# Show the first few rows
data.head()

### 2. Adding Indicators (Moving Average & RSI)
Moving averages smooth out the price data to show a trend. RSI tells us if a stock is overbought or oversold.

In [ ]:
# Calculate 50-day and 200-day Moving Averages
data['MA_50'] = data['Close'].rolling(window=50).mean()
data['MA_200'] = data['Close'].rolling(window=200).mean()

# Function to calculate RSI
def calculate_rsi(df, window=14):
    delta = df['Close'].diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    rsi = 100 - (100 / (1 + rs))
    return rsi

data['RSI'] = calculate_rsi(data)

# Drop NaN values created by rolling windows
data = data.dropna()

# Plotting the data with Moving Averages
plt.figure(figsize=(14, 7))
plt.plot(data.index, data['Close'], label='Close Price', color='blue')
plt.plot(data.index, data['MA_50'], label='50-day MA', color='red')
plt.plot(data.index, data['MA_200'], label='200-day MA', color='green')
plt.title(f'{stock_symbol} Stock Price with Moving Averages')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.legend()
plt.show()

### 3. Prepare Data for the LSTM Model
Neural networks work better when data is scaled between 0 and 1. I'll use MinMaxScaler for this.

In [ ]:
# Get just the 'Close' price
dataset = data.filter(['Close']).values

# Use 80% of data for training
training_data_len = math.ceil(len(dataset) * 0.8)

# Scale the data
scaler = MinMaxScaler(feature_range=(0,1))
scaled_data = scaler.fit_transform(dataset)

# Create training data set
train_data = scaled_data[0:training_data_len, :]

# Split into x_train and y_train
x_train = []
y_train = []

# Look back 60 days to predict the next day
look_back = 60
for i in range(look_back, len(train_data)):
    x_train.append(train_data[i-look_back:i, 0])
    y_train.append(train_data[i, 0])

# Convert to numpy arrays
x_train, y_train = np.array(x_train), np.array(y_train)

# Reshape the data for LSTM (samples, time steps, features)
x_train = np.reshape(x_train, (x_train.shape[0], x_train.shape[1], 1))

### 4. Build and Train the LSTM Model
I'm creating a simple sequential model with 2 LSTM layers.

In [ ]:
model = Sequential()
model.add(LSTM(50, return_sequences=True, input_shape=(x_train.shape[1], 1)))
model.add(LSTM(50, return_sequences=False))
model.add(Dense(25))
model.add(Dense(1))

# Compile the model
model.compile(optimizer='adam', loss='mean_squared_error')

# Train the model (using 5 epochs to save time)
print("Training model...")
model.fit(x_train, y_train, batch_size=32, epochs=5)

# Save the model weights so we can use it in our Streamlit app!
model.save('student_lstm_model.h5')
print("Model saved successfully to student_lstm_model.h5")

### 5. Testing the Model
Let's see how well the model predicts the remaining 20% of the data.

In [ ]:
# Create the testing data set
test_data = scaled_data[training_data_len - look_back: , :]

x_test = []
y_test = dataset[training_data_len:, :]

for i in range(look_back, len(test_data)):
    x_test.append(test_data[i-look_back:i, 0])

# Convert data to numpy array
x_test = np.array(x_test)

# Reshape data
x_test = np.reshape(x_test, (x_test.shape[0], x_test.shape[1], 1))

# Get the models predicted price values 
predictions = model.predict(x_test)
predictions = scaler.inverse_transform(predictions)

# Get the root mean squared error (RMSE)
rmse = np.sqrt(np.mean(((predictions - y_test) ** 2)))
print(f"Root Mean Squared Error: {rmse}")

### 6. Plotting the Predictions vs Actual Prices

In [ ]:
# Plot the data
train = data.filter(['Close'])[:training_data_len]
valid = data.filter(['Close'])[training_data_len:]
valid['Predictions'] = predictions

# Visualize the data
plt.figure(figsize=(14, 7))
plt.title('LSTM Model Predictions vs Actual Data')
plt.xlabel('Date')
plt.ylabel('Close Price USD ($)')
plt.plot(train.index, train['Close'])
plt.plot(valid.index, valid[['Close', 'Predictions']])
plt.legend(['Training Data', 'Actual Value', 'Predicted Value'], loc='lower right')
plt.show()